# Autograd - Build
This is the build section of the project, where new functions, classes, etc are created and tested. Finished peices are moved into main where they are then called and used by build.

### Rules:
- No ai used to generate any code, only for reaserch, explinations and code reviews
- Finished peices must be abstract and state clearly if any cases are not covered

## Setup

In [60]:
import numpy as np
import main
from typing import Union, List
from matplotlib import pyplot as plt
import matplotlib_inline
import pandas as pd
%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


## Build

In [61]:
a = [1, 2, 3, 4]
x = a[-2:]
for i in range(1, 4+1):
    print(i)

1
2
3
4


In [62]:
class Model():
    """
    
    """
    def __init__(self,
                  input_size : int, hidden_size : int, output_size : int,
                  number_of_layers : int, activation_function, normalisation_function,
                  precision: str = 'float32', random_seed: int = None):
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.number_of_layers = number_of_layers

        self.precision = precision
        self.random_seed = random_seed

        self.activation_function = activation_function
        self.normalisation_function = normalisation_function

        self.activation_function_derivative = main.derivatives.get(
            getattr(self.activation_function, "__name__", None)
        )
        self.normalisation_function_derivative = main.derivatives.get(
            getattr(self.normalisation_function, "__name__", None)
        )


        layers = []
        layer_gradients = {}
        if self.number_of_layers == 1:
            layers.append([main.LinearLayer(self.input_size, self.output_size,
                                            self.precision, self.random_seed)])
            layer_gradients["Layer 1"] = {}
        else:
            layers.append([main.LinearLayer(self.input_size, self.hidden_size,
                                            self.precision, self.random_seed)])
            layer_gradients["Layer 1"] = {}

            for i in range(max(0, self.number_of_layers - 2)):
                layers.append([
                    main.LinearLayer(self.hidden_size, self.hidden_size,
                                     self.precision, self.random_seed),
                    self.activation_function
                ])
                layer_gradients[f"Layer {i+1}"] = {}

            layers.append([
                main.LinearLayer(self.hidden_size, self.output_size,
                                 self.precision, self.random_seed)
            ])
            layer_gradients[f"Layer {self.number_of_layers}"] = {}

        layers.append([self.normalisation_function])

        self.modules = layers
        # self.gradients serves no functional purposse, but make it easier to acces all the differnt layer's gradients at once.
        self.gradients = layer_gradients

        parramaters = {}
        i = 1
        for obj in self.modules:
            for sub_obj in obj:
                if isinstance(sub_obj, main.LinearLayer):
                    parramaters[f"Layer {i}"] = sub_obj.parramaters
                    i += 1

        self.parramaters = parramaters

        self.last_operation = None


    def forward(self, x: Union[np.ndarray, List, float, int]) -> np.ndarray:
        for module in self.modules:
            for obj in module:
                if hasattr(obj, "forward") and callable(obj.forward):
                    x = obj.forward(x)
                elif callable(obj):
                    x = obj(x)
                else:
                    raise TypeError(f"Module - {type(obj).__name__} - does not have a 'forward' function, or is not callable and so is not supported")
        self.last_operation = "forward"
        return x


    def backwards(self, output : np.ndarray, target : np.ndarray, loss_function):
        """
        """
        if self.last_operation != "forward":
            raise main.SequenceError("Must compleate a forwards pass imidatley before a backwards pass.")

        try:
            output = np.asarray(output, dtype=self.precision)
            target = np.asarray(target, dtype=self.precision)
        except (TypeError, ValueError) as e:
            raise TypeError(f"The inputed arrays - {output} and {target} must be a numeric array or array-like object. Original error: {e}")

        # Check for empty arrays
        if output.size == 0 or target.size == 0:
            raise ValueError("At least one input is empty.")
        
        if output.ndim == 1:
            output = np.array([output], dtype=self.precision)

        if target.ndim == 1:
            target = np.array([target], dtype=self.precision)
        
        
        # As the partial derivative of loss with respect to pre-normalised-output for Softmax then CrossEntropyLoss is
        # targets - pre_normalisation_outputs (a much simpler formular than the deviratives of either function) it make
        # sence to check for this.
        if self.modules[-1] == [main.Softmax] and loss_function == main.CrossEntropyLoss:
            # passed_down_grad stores the previous layers desired output to be compared aginst.
            passed_down_gard = output - target
        else:
            raise TypeError(f"This model's normalisation function - {self.normalisation_function} - currently has no programed back propogtion rules.")


        # Backwards pass thorugh the final layer
        layer = self.modules[-2][0]  # last linear layer before softmax
        layer.backwards(passed_down_gard)
        passed_down_gard = layer.passed_down_grad
        self.gradients[f"Layer {self.number_of_layers}"] = layer.gradients


        # Backwards pass through hidden layers
        for layer_index in range(self.number_of_layers - 1, 0, -1):
            layer_modual = self.modules[layer_index - 1]
            layer = layer_modual[0]

            layer.backwards(passed_down_gard)
            passed_down_gard = layer.passed_down_grad

            # Check there is an activation function in the modual before trying to go backwards through it.
            if len(layer_modual) > 1:
                if self.activation_function is main.ReLU:
                    passed_down_gard = passed_down_gard * self.activation_function_derivative(layer.last_outputed_array)
                else:
                    raise TypeError(f"This model's activation function - {self.activation_function} - currently has no programed back propogtion rules.")

            self.gradients[f"Layer {layer_index}"] = layer.gradients

        self.last_operation = "backwards"
    

    def update_parramaters(self, learning_rate : float):
        """
        """
        if self.last_operation != "backwards":
            raise main.SequenceError("Must compleate a backwards pass imidatley before updating a models parramaters.")

        try: 
            float(learning_rate)
        except (TypeError, ValueError) as exc:
            raise TypeError("Learning rate must be a number, preferably float") from exc

        if not np.isfinite(learning_rate):
            raise ValueError(f"Learning rate must be finite, currently it is - {learning_rate}")

        
        for layer_index in range(self.number_of_layers, 0, -1):
            layer_modual = self.modules[layer_index -1]
            layer = layer_modual[0]
            layer.update_parameters(learning_rate)

        self.last_operation = "update"
        

    def set_parramaters(self, parramaters : dict):
        """
        Sets the model parameters to the ones stored in the input dictionary.

        Args:
            parramaters (dict): The dictionary storing the parramaters.

        Updates:
            self.parramaters: Sets self.parramaters["Layer i"] to parramaters["Layer i"]

        Rasies:
            TypeError: If the input is not a dictionary.
            KeyError: If the inputed dictionary does not have the keys "Layer 1" thorugh "Layer i".
        """

        if not isinstance(parramaters, dict):
            raise TypeError(f"Input - {parramaters} - must be a dict.")

        if set(parramaters.keys()) != set(self.parramaters.keys()):
            raise KeyError(f"Input - {parramaters} - keys do not match those in self.parramaters.")

        if len(self.parramaters) != len(parramaters):
            raise ValueError(f"Inputed dictionary has a different number of elements than the existing dictionary.")

        
        modules_copy = self.modules
        layer_index = 1
        for obj in modules_copy:
            if isinstance(obj, main.LinearLayer):
                key = f"Layer {layer_index}"
                if key not in parramaters:
                    raise KeyError(f"Missing parameter set for {key}.")
                obj.set_parramaters(parramaters[key])
                layer_index += 1

        self.modules = modules_copy
        self.parramaters = parramaters
    

In [ ]:
test_inputs = [
    np.array([-1, -0.5, 0]),
    [[0, 1, 2], [2, 0.7, -1]],
    (-2, -0.32, 0),
    "This is a string."
]

model = Model(input_size=3, output_size=2, hidden_size=2,
              number_of_layers=4, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=1)

In [64]:
print(model.modules)

[[<main.LinearLayer object at 0x000001E9F1184190>], [<main.LinearLayer object at 0x000001E9F1185310>, <function ReLU at 0x000001E9ED77D940>], [<main.LinearLayer object at 0x000001E9F239ECB0>, <function ReLU at 0x000001E9ED77D940>], [<main.LinearLayer object at 0x000001E9F239EE90>], [<function Softmax at 0x000001E9ED77DA80>]]


In [65]:
for t in test_inputs:
    try:
        print(f"Input - {t} - Output - {model.forward(t)}")
        print()
    except:
        print(f"Input - {t} - can not be passed through the model.")

Input - [-1.  -0.5  0. ] - Output - [[0.44546778 0.55453222]]

Input - [[0, 1, 2], [2, 0.7, -1]] - Output - [[0.44546778 0.55453222]
 [0.39431958 0.60568042]]

Input - (-2, -0.32, 0) - Output - [[0.44546778 0.55453222]]

Input - This is a string. - can not be passed through the model.


In [109]:
#Organising the data
data = pd.read_csv(r"C:\Users\acart\OneDrive\Desktop\CodingProjects\mnist\mnist_train.csv")

data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:30000]
Y_dev = data_dev[0].T
X_dev = data_dev[1:n]
X_dev = X_dev / 255.

data_train = data[30000:m]
print(data_train.shape)
Y_train = data_train[0].T
X_train = data_train[1:n]
X_train = X_train / 255.
_,m_train = X_train.shape

(29999, 785)


In [110]:
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

Y_dev, Y_train = one_hot(Y_dev), one_hot(Y_train)

In [113]:
print(X_train.shape)
print(Y_train.shape)
print(Y_train[0])


(784, 785)
(256, 785)
[0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0.
 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1

In [112]:
model = Model(input_size=785, output_size=10, hidden_size=16,
              number_of_layers=4, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=1)

for i in range(100):
    output = model.forward(X_train)
    loss = main.CrossEntropyLoss(output, Y_train)

ValueError: Input and target arrays must have the same shape.